In [ ]:
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.optimizers import Adam

# ۱. تنظیمات مسیرها و لیست سنسورها
file_path = r'second_stage_inputs\G11\dsas_g11_bearings_vibration_temp_output.xlsx'
output_path = r'outputs\G11\dsas_g11_bearings_vibration_temp_deviation_monitoring\deviation_monitoring\autoencoder_keras_output.xlsx'

all_sensors = ['AssetID_9357', 'AssetID_9343', 'AssetID_9358', 'AssetID_9359', 
               'AssetID_9360', 'AssetID_9361', 'AssetID_9368', 'AssetID_9369', 'AssetID_9370']

# ۲. بارگذاری داده‌ها
if not os.path.exists(file_path):
    print(f"❌ فایل یافت نشد: {file_path}")
else:
    df = pd.read_excel(file_path)
    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values(by='date')

    # پالایش داده‌های بدون مقدار
    df_clean = df.dropna(subset=all_sensors).copy()
    raw_data = df_clean[all_sensors].values

    # ۳. نرمال‌سازی (بسیار مهم برای همگرایی مدل‌های عمیق)
    scaler = MinMaxScaler()
    scaled_data = scaler.fit_transform(raw_data)

    # ۴. ساخت معماری اتوانکودر با Keras
    input_dim = len(all_sensors)

    # تعریف لایه‌ها
    input_layer = Input(shape=(input_dim,))
    # Encoder: فشرده‌سازی داده‌ها برای استخراج روابط سیستماتیک
    encoded = Dense(16, activation='relu')(input_layer)
    latent_space = Dense(8, activation='relu')(encoded) # فضای فشرده پنهان

    # Decoder: بازسازی مجدد داده‌ها از فضای فشرده
    decoded = Dense(16, activation='relu')(latent_space)
    output_layer = Dense(input_dim, activation='sigmoid')(decoded) # خروجی با ابعاد ورودی

    # ایجاد مدل کل
    autoencoder = Model(inputs=input_layer, outputs=output_layer)
    autoencoder.compile(optimizer=Adam(learning_rate=0.01), loss='mse')

    # ۵. آموزش مدل
    print("🚀 شروع آموزش مدل اتوانکودر (Keras) برای شناسایی رفتارهای سیستماتیک...")
    autoencoder.fit(
        scaled_data, scaled_data,
        epochs=150,
        batch_size=32,
        shuffle=True,
        verbose=0 # برای خلوت ماندن کنسول، خروجی لاگ‌ها خاموش شده
    )

    # ۶. محاسبه بازسازی و خطای آن
    reconstructed_data = autoencoder.predict(scaled_data)

    # محاسبه MSE برای هر ردیف (میزان انحراف سیستم از وضعیت نرمال)
    # تفاوت توان دوم بین داده واقعی و بازسازی شده
    mse_errors = np.mean(np.power(scaled_data - reconstructed_data, 2), axis=1)

    # ۷. تعیین آستانه ناهنجاری (Threshold) بر اساس متد ۳ سیگما
    threshold = np.mean(mse_errors) + (3  np.std(mse_errors))

    # ۸. اضافه کردن نتایج به دیتافریم اصلی
    df_clean['Systematic_Deviation_Index'] = mse_errors
    df_clean['Anomaly_Threshold'] = threshold
    df_clean['Is_Anomaly'] = df_clean['Systematic_Deviation_Index'] > threshold

    # ۹. ذخیره خروجی
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    df_clean.to_excel(output_path, index=False)

    print("-" * 40)
    print(f"✅ تحلیل با موفقیت انجام شد.")

    print(f"📍 تعداد کل داده‌های پردازش شده: {len(df_clean)}")
    print(f"🚨 تعداد موارد ناهنجار شناسایی شده: {df_clean['Is_Anomaly'].sum()}")
    print(f"📂 فایل خروجی در مسیر زیر ذخیره شد:\n{output_path}")

تعداد کل داده‌های آموزشی: 11633
تعداد داده‌های سالم تایید شده برای آموزش: 10946
میزان داده‌های پرت حذف شده: 687 ردیف
پایش با مدل پالایش شده انجام و در outputs\G11\dsas_g11_bearings_vibration_temp_deviation_monitoring\deviation_monitoring\dsas_g11_bearings_vibration_temp_deviation_monitoring_output3.xlsx ذخیره شد.
